In [1]:
!pip install transformers datasets torch scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from pathlib import Path

from transformers import EarlyStoppingCallback

In [3]:
possible_paths = [
    Path('../dataset/dataset_clean_final.csv'),
    Path('dataset_clean_final.csv'),
    Path('/content/dataset_clean_final.csv')
]

for path in possible_paths:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Dataset berhasil dibaca dari: {path}")
        break
else:
    raise FileNotFoundError('File dataset_clean_final.csv tidak ditemukan. Pastikan file berada di folder dataset atau satu folder dengan notebook.')

Dataset berhasil dibaca dari: dataset_clean_final.csv


In [4]:
# Load dataset
df = df.dropna(subset=['text', 'label'])
df.head()

,text,label
0,makin yakin habis baca review lain tentang vic...,1
1,paling suka model h2 smiling_face_with_heart e...,0
2,mobilnya sudah hancur pleading_face,0
3,manut88benar2 bikin aku jadi sultan,1
4,semoga lekas recover mobilnya mas dipo,0


In [5]:
# Membagi dataset menjadi 80% Training dan 20% Validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'] # Menjaga rasio label tetap seimbang
)

# Konversi format list biasa menjadi format 'Dataset' bawaan Hugging Face
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

print(f"Jumlah data latih: {len(train_dataset)}")
print(f"Jumlah data validasi: {len(val_dataset)}")

Jumlah data latih: 56303
Jumlah data validasi: 14076


In [6]:
# Menggunakan pre-trained model IndoBERT
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # max_length=128 cukup ideal untuk komentar YouTube
    # padding & truncation memastikan semua input memiliki panjang seragam
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Menerapkan tokenisasi ke seluruh data latih dan validasi secara massal (batched)
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("Proses tokenisasi selesai!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/56303 [00:00<?, ? examples/s]

Map:   0%|          | 0/14076 [00:00<?, ? examples/s]

Proses tokenisasi selesai!


In [7]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Menghitung metrik klasifikasi biner
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

# 1. Definisi Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Hitung Cross Entropy standar
        CE_loss = F.cross_entropy(inputs, targets, reduction='none')

        # Probabilitas prediksi (pt)
        pt = torch.exp(-CE_loss)

        # Focal Modulator: (1 - pt)^gamma
        # Ini mengecilkan loss dari sampel yang MUDAH ditebak (kelas mayoritas)
        # dan membesarkan loss dari sampel yang SULIT (slang judol seperti 'wd', 'gacor')
        F_loss = ((1 - pt) ** self.gamma) * CE_loss

        # Aplikasikan Class Weight (alpha)
        if self.alpha is not None:
            at = self.alpha.gather(0, targets)
            F_loss = at * F_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        return F_loss

# 2. Definisi Custom Trainer
class WeightedFocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Bobot Alpha: Kelas 0 = 1.0, Kelas 1 = 7.61 (Rasio 62227 / 8177)
        # Pindahkan tensor ke device yang sama dengan model (CPU/GPU)
        alpha = torch.tensor([1.0, 7.61], dtype=torch.float).to(logits.device)

        # Gamma = 2.0 adalah standar empiris untuk Focal Loss
        loss_fct = FocalLoss(alpha=alpha, gamma=2.0, reduction='mean')
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [9]:
# Memuat model IndoBERT dengan spesifikasi 2 label kelas klasifikasi
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Konfigurasi argumen pelatihan
training_args = TrainingArguments(
    output_dir="./indobert_judol_model_focal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# Inisialisasi API Trainer
trainer = WeightedFocalTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [10]:
# Memulai proses training
print("Memulai proses fine-tuning IndoBERT...")
trainer.train()

Memulai proses fine-tuning IndoBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.061964,0.030240,0.996022,0.982769,0.988854,0.976758
2,0.034463,0.034716,0.996022,0.982822,0.985846,0.979817
3,0.011306,0.046868,0.996377,0.984332,0.988889,0.979817
4,0.001768,0.057460,0.995311,0.979927,0.974592,0.985321
5,0.009667,0.061790,0.996732,0.985898,0.988322,0.983486


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17595, training_loss=0.02548399055598298, metrics={'train_runtime': 6989.8495, 'train_samples_per_second': 40.275, 'train_steps_per_second': 2.517, 'total_flos': 1.85174271874176e+16, 'train_loss': 0.02548399055598298, 'epoch': 5.0})

In [11]:
# Menentukan nama folder penyimpanan
model_path = "./model/indobert_judol_model_focal"

# Menyimpan model dan tokenizer
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model berhasil disimpan di folder: {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder: ./model/indobert_judol_model_focal


In [12]:
from transformers import pipeline

# Load model yang baru saja disimpan
classifier = pipeline("text-classification", model=model_path, tokenizer=model_path)

# Contoh teks pengujian
test_comments = [
    "Wah videonya sangat edukatif, terima kasih bang!",
    "Bongkar rahasia wd terus bosku, cek link di bio sekarang depo 10k jadi 100k",
    "Gacor banget bang mainnya, tutor dong"
]

# Jalankan prediksi
predictions = classifier(test_comments)

for text, pred in zip(test_comments, predictions):
    label = "JUDOL" if pred['label'] == 'LABEL_1' else "BUKAN JUDOL"
    print(f"Komentar: {text}")
    print(f"Prediksi: {label} (Confidence: {pred['score']:.4f})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Komentar: Wah videonya sangat edukatif, terima kasih bang!
Prediksi: BUKAN JUDOL (Confidence: 0.9947)

Komentar: Bongkar rahasia wd terus bosku, cek link di bio sekarang depo 10k jadi 100k
Prediksi: JUDOL (Confidence: 0.9766)

Komentar: Gacor banget bang mainnya, tutor dong
Prediksi: BUKAN JUDOL (Confidence: 0.9871)

